In [1]:
# Cell 1: Imports & Configurations
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dotenv import load_dotenv
from fredapi import Fred
import timesfm

# Load environment variables
load_dotenv()

# Securely fetch the FRED API key
FRED_API_KEY = os.getenv('FRED_API_KEY')
if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not found. Please ensure your .env file is set up correctly.")

# Initialize FRED API client
fred = Fred(api_key=FRED_API_KEY)

# Configure clean, corporate plotting styles for business analysts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.autolayout': True,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'lines.linewidth': 2.5,
    'font.family': 'sans-serif',
    'legend.frameon': True,
    'legend.facecolor': 'white'
})

print("✅ Libraries imported, styling configured, and FRED API initialized successfully.")

✅ Libraries imported, styling configured, and FRED API initialized successfully.


In [2]:
# Cell 2: Model Instantiation
import torch
import timesfm

print("Initializing TimesFM 2.5 model...")

# Set matmul precision for hardware acceleration (Ampere+ GPUs)
torch.set_float32_matmul_precision("high")

# Define context and horizon based on our 12-month data plan
CONTEXT_LENGTH = 300
HORIZON_LENGTH = 12

# Instantiate the 2.5 model directly from HuggingFace
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

# Compile the model with the correct forecast configuration
tfm.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT_LENGTH,
        max_horizon=HORIZON_LENGTH,
        normalize_inputs=True,              # Recommended to prevent scale instability
        use_continuous_quantile_head=True,  # Recommended for accurate quantiles
        return_backcast=True
    )
)

print(f"✅ TimesFM 2.5 instantiated and compiled successfully with a {HORIZON_LENGTH}-month horizon.")

Initializing TimesFM 2.5 model...


✅ TimesFM 2.5 instantiated and compiled successfully with a 12-month horizon.


In [3]:
# Cell 3: Load Data & Convert to Long Format
print("Loading and reshaping dataset...")

# Define the exact filename verbatim
file_path = "U20405-M - Sheet1.csv"

try:
    # Read the wide-format CSV file
    df_wide = pd.read_csv(file_path)
    print(f"✅ Successfully loaded '{file_path}' in wide format.")
    
    # Melt the dataframe from wide to long
    df_long = pd.melt(
        df_wide, 
        id_vars=['category', 'category_key'], # Keep identifiers intact
        var_name='Date_Raw',                  # The previous column headers (e.g., '1959M01')
        value_name='value'                   # The actual index values
    )
    
    # Convert the BEA 'YYYYMmm' format into proper pandas datetime objects
    # We replace 'M' with '-' to make it 'YYYY-mm', which pandas parses natively
    df_long['Date'] = pd.to_datetime(df_long['Date_Raw'].str.replace('M', '-'))
    
    # Drop the raw date string to keep it clean, sort chronologically, and drop missing values
    df_long = df_long.drop(columns=['Date_Raw']).sort_values(by=['category_key', 'Date']).dropna(subset=['value']).reset_index(drop=True)
    
    print(f"✅ Dataset successfully reshaped to long format.")
    print(f"New dataset shape: {df_long.shape[0]} rows, {df_long.shape[1]} columns.\n")
    
    # Display the first few rows to verify the transformation
    display(df_long.head())
    
except FileNotFoundError:
    print(f"❌ Error: Could not find '{file_path}'. Please ensure the file is in the same directory.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

Loading and reshaping dataset...
✅ Successfully loaded 'U20405-M - Sheet1.csv' in wide format.
✅ Dataset successfully reshaped to long format.
New dataset shape: 324414 rows, 4 columns.



,category,category_key,value,Date
0,Expenditures abroad by U.S. residents,DABDRC,"1,082",1959-01-01
1,Expenditures abroad by U.S. residents,DABDRC,"1,108",1959-02-01
2,Expenditures abroad by U.S. residents,DABDRC,"1,146",1959-03-01
3,Expenditures abroad by U.S. residents,DABDRC,"1,130",1959-04-01
4,Expenditures abroad by U.S. residents,DABDRC,"1,075",1959-05-01


In [4]:
# %%
# Cell 4: Clean & Align Multiple Variables
print("Cleaning and aligning datasets...")

# 1. Process all 4 BEA indices from the reshaped long dataset
target_keys = ['DVAPRC', 'DFSARC', 'DTRSRC', 'DRCARC', 'DGOERC', 'DPCERC']
df_targets = df_long[df_long['category_key'].isin(target_keys)].copy()
df_targets['Date'] = pd.to_datetime(df_targets['Date'])

# FIX: Use pivot_table with aggfunc='first' to gracefully handle any duplicate rows in the raw CSV
df_targets = df_targets.pivot_table(
    index='Date', 
    columns='category_key', 
    values='value', 
    aggfunc='first'
)
df_targets = df_targets.resample('MS').first() # Force strict monthly frequency

# 2. Fetch Brent Crude dynamically from FRED
print("Fetching Brent Crude (POILBREUSDM) from FRED API...")
brent_series = fred.get_series('POILBREUSDM')
df_brent = pd.DataFrame({'Brent_Crude': brent_series})
df_brent.index.name = 'Date'
df_brent.index = pd.to_datetime(df_brent.index)
df_brent = df_brent.resample('MS').mean()

# 3. Merge the datasets
df_clean = pd.merge(df_targets, df_brent, left_index=True, right_index=True, how='outer')

# 4. Handle gaps and trimming
df_clean['Brent_Crude'] = df_clean['Brent_Crude'].ffill()

# Trim to when we have data for the BEA indices
first_valid_date = df_clean['DVAPRC'].first_valid_index()
df_clean = df_clean.loc[first_valid_date:]

# Ragged Edge preservation
df_clean = df_clean[df_clean['Brent_Crude'].notna()]

print("✅ Data aligned successfully. Multi-variate structure established.")
display(df_clean.tail(3))

Cleaning and aligning datasets...


Fetching Brent Crude (POILBREUSDM) from FRED API...
✅ Data aligned successfully. Multi-variate structure established.


,DFSARC,DGOERC,DPCERC,DRCARC,DTRSRC,DVAPRC,Brent_Crude
Date,,,,,,,
2026-01-01,"1,514,584","415,724","21,525,465","859,038","728,267","446,165",64.594091
2026-02-01,"1,522,911","422,429","21,665,077","850,537","744,523","451,196",69.409500
2026-03-01,"1,526,367","503,683","21,860,518","856,019","751,261","455,896",99.405000


In [5]:
# %%
# Cell 5: Extract High-Oil Historical Scenarios
import numpy as np
import pandas as pd

print("Identifying >$100 Brent Crude periods & extracting scenario trajectories...")

# 1. BULLETPROOF NUMERIC CONVERSION
for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.replace(',', '', regex=False).str.strip()
    
df_clean = df_clean.apply(pd.to_numeric, errors='coerce')

TARGET_SECTOR = 'DVAPRC'
intermediate_covariates = [col for col in df_clean.columns if col not in [TARGET_SECTOR, 'Brent_Crude']]
print(f"✅ Dynamic Macro Drivers identified: {intermediate_covariates}")

# 2. Programmatically find the start of >$100 oil shocks
brent_above_100 = df_clean['Brent_Crude'] > 100
start_of_spikes = brent_above_100 & ~brent_above_100.shift(1, fill_value=False)
anchor_dates = df_clean.index[start_of_spikes]

# 3. Extract 12-month multipliers distinctively for EACH scenario
scenario_shock_shapes = {}

print(f"Found {len(anchor_dates)} historical oil shocks crossing $100:")
for anchor in anchor_dates:
    scenario_name = anchor.strftime('%Y-%b')
    print(f" - Initializing Scenario: {scenario_name}")
    
    end_date = anchor + pd.DateOffset(months=12)
    scenario_shock_shapes[scenario_name] = {}
    
    for cov in intermediate_covariates:
        valid_cov_data = df_clean[cov].dropna()
        
        # SAFE LOOP: Ensure data exists for this specific historical timeframe
        if not valid_cov_data.empty and end_date <= valid_cov_data.index[-1]:
            slice_13m = df_clean.loc[anchor:end_date, cov]
            
            if len(slice_13m) >= 13: 
                trajectory = (slice_13m.values / slice_13m.values[0])[1:]
                scenario_shock_shapes[scenario_name][cov] = trajectory
            else:
                scenario_shock_shapes[scenario_name][cov] = np.ones(12) # Fallback
        else:
            scenario_shock_shapes[scenario_name][cov] = np.ones(12) # Fallback

print("✅ Scenario extraction complete.")

Identifying >$100 Brent Crude periods & extracting scenario trajectories...
✅ Dynamic Macro Drivers identified: ['DFSARC', 'DGOERC', 'DPCERC', 'DRCARC', 'DTRSRC']
Found 4 historical oil shocks crossing $100:
 - Initializing Scenario: 2008-Mar
 - Initializing Scenario: 2011-Feb
 - Initializing Scenario: 2012-Jul
 - Initializing Scenario: 2022-Mar
✅ Scenario extraction complete.


In [6]:
# %%
# Cell 6: Forecast Intermediaries per Scenario
print("Executing scenario-based forecasts for intermediate covariates...")

HISTORICAL_WEIGHT = 0.60  
TIMESFM_WEIGHT = 1.0 - HISTORICAL_WEIGHT
print(f"Weighting Strategy: {HISTORICAL_WEIGHT*100}% Scenario History / {TIMESFM_WEIGHT*100}% TimesFM Baseline")

future_dates = pd.date_range(start=df_clean.index[-1] + pd.DateOffset(months=1), periods=HORIZON_LENGTH, freq='MS')

# 1. Pre-compute TimesFM Baselines (Only needs to run once)
timesfm_baselines = {}
baseline_values = {}

for cov in intermediate_covariates:
    history_array = df_clean[cov].dropna().values
    baseline_values[cov] = history_array[-1] # Save the jump-off point
    context_data = history_array[-CONTEXT_LENGTH:]
    
    point_forecast, _ = tfm.forecast(horizon=HORIZON_LENGTH, inputs=[context_data])
    timesfm_baselines[cov] = point_forecast[0][-HORIZON_LENGTH:]

# 2. Blend the baseline with EACH scenario
scenario_cov_forecasts = {}

for scenario, shapes in scenario_shock_shapes.items():
    df_scenario = pd.DataFrame(index=future_dates)
    
    for cov in intermediate_covariates:
        pure_historical = baseline_values[cov] * shapes[cov]
        blended_forecast = (pure_historical * HISTORICAL_WEIGHT) + (timesfm_baselines[cov] * TIMESFM_WEIGHT)
        df_scenario[cov] = blended_forecast
        
    scenario_cov_forecasts[scenario] = df_scenario

print(f"\n✅ {HORIZON_LENGTH}-Month Covariate Forecasts generated for {len(scenario_cov_forecasts)} scenarios.")

Executing scenario-based forecasts for intermediate covariates...
Weighting Strategy: 60.0% Scenario History / 40.0% TimesFM Baseline

✅ 12-Month Covariate Forecasts generated for 4 scenarios.


In [7]:
# %%
# Cell 7: Execute Target XReg Forecast Iteratively
import numpy as np
print(f"Executing Multi-Scenario Target Forecasts for: {TARGET_SECTOR}...")

target_series = df_clean[TARGET_SECTOR].dropna()
target_last_date = target_series.index[-1]
target_context = target_series.values[-CONTEXT_LENGTH:]
actual_context_len = len(target_context)
timesfm_target_input = [target_context.astype(np.float32)]

scenario_target_forecasts = {}

for scenario, df_cov_forecast in scenario_cov_forecasts.items():
    dynamic_cov_dict = {}
    
    # Build dynamic covariates for THIS scenario
    for cov in intermediate_covariates:
        cov_history = df_clean.loc[:target_last_date, cov].ffill()
        cov_context = cov_history.values[-actual_context_len:]
        
        cov_future_values = []
        for d in future_dates:
            if d in df_clean.index and not pd.isna(df_clean.loc[d, cov]):
                cov_future_values.append(df_clean.loc[d, cov])
            elif d in df_cov_forecast.index:
                cov_future_values.append(df_cov_forecast.loc[d, cov])
            else:
                cov_future_values.append(cov_future_values[-1] if cov_future_values else cov_context[-1])
                
        cov_future = np.array(cov_future_values)
        full_cov_array = np.concatenate([cov_context, cov_future])
        dynamic_cov_dict[cov] = [full_cov_array.astype(np.float32)]

    # Forecast Target using this scenario's exogenous features
    point_forecast, _ = tfm.forecast_with_covariates(
        inputs=timesfm_target_input,
        dynamic_numerical_covariates=dynamic_cov_dict,
        xreg_mode="xreg + timesfm" 
    )
    
    # Store Results
    df_target_forecast = pd.DataFrame(index=future_dates)
    df_target_forecast[f'{TARGET_SECTOR}_Forecast'] = point_forecast[0]
    scenario_target_forecasts[scenario] = df_target_forecast

print(f"✅ Executed {len(scenario_target_forecasts)} distinct scenario forecasts.")

Executing Multi-Scenario Target Forecasts for: DVAPRC...
✅ Executed 4 distinct scenario forecasts.


In [17]:
# %%
# Cell 8: Interactive Scenario Dashboards (Dynamic Names)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("Generating Interactive Plotly Dashboards...")

historical_slice = df_clean.loc['2005-01-01':].copy()
target_last_date = df_clean[TARGET_SECTOR].dropna().index[-1]
last_target_val = df_clean[TARGET_SECTOR].dropna().iloc[-1]

# Pull the exact mapping from df_long to avoid any variable collisions
csv_names = dict(zip(df_long['category_key'], df_long['category']))
target_display_name = csv_names.get(TARGET_SECTOR, TARGET_SECTOR)

fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=(
        f"<b>{target_display_name} Forecast</b> (Scenario Sensitivity)",
        f"<b>Macroeconomic Drivers</b> (Scenario Forecasts)"
    )
)

# --- TOP CHART: HISTORICAL TARGET ---
fig.add_trace(go.Scatter(
    x=historical_slice.index, y=historical_slice[TARGET_SECTOR],
    mode='lines', name=f'Historical {target_display_name}', 
    line=dict(color='black', width=3), legendgroup='History'
), row=1, col=1)

# Generate a color palette for the scenarios
scenario_names = list(scenario_target_forecasts.keys())
colors = px.colors.qualitative.Plotly 

# --- TOP CHART: FORECAST SCENARIOS ---
for idx, scenario in enumerate(scenario_names):
    color = colors[idx % len(colors)]
    plot_target = scenario_target_forecasts[scenario].copy()
    plot_target.loc[target_last_date] = last_target_val # Bridge gap
    plot_target = plot_target.sort_index()
    
    fig.add_trace(go.Scatter(
        x=plot_target.index, y=plot_target[f'{TARGET_SECTOR}_Forecast'],
        mode='lines', name=f'{scenario} Scenario', 
        line=dict(color=color, width=2, dash='dash'), legendgroup=scenario
    ), row=1, col=1)

# --- BOTTOM CHART: MACRO DRIVER SCENARIOS ---
for cov in intermediate_covariates:
    display_name = csv_names.get(cov, cov)
    cov_last_date = df_clean[cov].dropna().index[-1]
    cov_last_val = df_clean[cov].dropna().iloc[-1]
    
    # Historical Covariates
    fig.add_trace(go.Scatter(
        x=historical_slice.index, y=historical_slice[cov],
        mode='lines', name=display_name, 
        line=dict(color='gray', width=1), showlegend=False, legendgroup='History'
    ), row=2, col=1)
    
    # Forecast Covariates per Scenario
    for idx, scenario in enumerate(scenario_names):
        color = colors[idx % len(colors)]
        plot_cov = scenario_cov_forecasts[scenario][[cov]].copy()
        plot_cov.loc[cov_last_date] = cov_last_val # Bridge gap
        plot_cov = plot_cov.sort_index()
        
        fig.add_trace(go.Scatter(
            x=plot_cov.index, y=plot_cov[cov],
            mode='lines', name=f'{display_name} ({scenario})', 
            line=dict(color=color, width=1.5, dash='dot'), 
            showlegend=False, legendgroup=scenario # Groups legends so clicking one toggles all related traces
        ), row=2, col=1)

# Formatting
fig.add_vline(x=target_last_date, line_width=2, line_dash="dot", line_color="black", row='all', col='all')

fig.update_layout(
    height=900, template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

Generating Interactive Plotly Dashboards...


In [11]:
# %%
# Cell 9: Export Data in Horizontal Wide-by-Date Format
import pandas as pd

print("Transforming historical and scenario forecast data into horizontal date format...")

# Define the nice names mapping again to populate the 'category' column
nice_names = {
    'DPCERC': 'Overall PCE', 'DFSARC': 'Food PCE', 'DTRSRC': 'Transportation PCE',
    'DVAPRC': 'Tech/Media Equipment PCE', 'DGOERC': 'Gasoline/Energy PCE', 'DRCARC': 'Recreation PCE', 
}

# Ensure we are pulling the right variables
cols_to_export = [TARGET_SECTOR] + intermediate_covariates

# 1. Process Historical Baseline
# Transpose so variables are rows and dates are columns
df_hist = df_clean[cols_to_export].copy().T
df_hist['scenario'] = 'Historical Baseline'

# 2. Process Each Forecast Scenario
scenario_dfs = []
for scenario_name in scenario_target_forecasts.keys():
    # Grab target and rename column to standard identifier
    df_tgt = scenario_target_forecasts[scenario_name].rename(
        columns={f'{TARGET_SECTOR}_Forecast': TARGET_SECTOR}
    )
    
    # Grab covariates
    df_cov = scenario_cov_forecasts[scenario_name]
    
    # Combine, transpose, and tag the scenario
    df_combined = pd.concat([df_tgt, df_cov], axis=1).T
    df_combined['scenario'] = scenario_name
    
    scenario_dfs.append(df_combined)

# 3. Concatenate Everything Together
df_final = pd.concat([df_hist] + scenario_dfs)

# 4. Restructure the Metadata Columns
df_final.index.name = 'category_key'
df_final = df_final.reset_index()

# Insert the plain English 'category' column at the very front
df_final.insert(0, 'category', df_final['category_key'].map(nice_names).fillna(df_final['category_key']))

# 5. Clean up Date Headers 
# Identify the date columns (everything that isn't our metadata)
date_cols = [col for col in df_final.columns if col not in ['category', 'category_key', 'scenario']]

# Format the datetime objects into clean strings (e.g., '2024-01', '2024-02')
formatted_date_cols = [d.strftime('%Y-%m') if pd.notna(d) else str(d) for d in date_cols]
rename_dict = dict(zip(date_cols, formatted_date_cols))
df_final = df_final.rename(columns=rename_dict)

# Force the exact column order requested
final_col_order = ['category', 'category_key', 'scenario'] + formatted_date_cols
df_final = df_final[final_col_order]

# Sort neatly by category_key, making sure Historical Baseline appears before Forecasts
df_final['sort_order'] = df_final['scenario'].apply(lambda x: 0 if x == 'Historical Baseline' else 1)
df_final = df_final.sort_values(by=['category_key', 'sort_order', 'scenario']).drop(columns=['sort_order'])

# 6. Export to CSV
export_filename = f"{TARGET_SECTOR}_Horizontal_Forecasts.csv"
df_final.to_csv(export_filename, index=False)

print(f"✅ Data successfully exported to '{export_filename}'")
print(f"Rows: {len(df_final)} | Columns: {len(df_final.columns)}")

# Display a preview of the new structure
display(df_final.head(10))

Transforming historical and scenario forecast data into horizontal date format...
✅ Data successfully exported to 'DVAPRC_Horizontal_Forecasts.csv'
Rows: 30 | Columns: 426


,category,category_key,scenario,1992-01,1992-02,1992-03,1992-04,1992-05,1992-06,1992-07,...,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12,2027-01,2027-02,2027-03
1,Food PCE,DFSARC,Historical Baseline,284955.0,285438.0,284833.0,281367.0,282989.0,277513.0,280546.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Food PCE,DFSARC,2008-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.542997e+06,1.545055e+06,1.548926e+06,1.545541e+06,1.548112e+06,1.544602e+06,1.545605e+06,1.544350e+06,1.543561e+06,1.531546e+06
13,Food PCE,DFSARC,2011-Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.543078e+06,1.550787e+06,1.554148e+06,1.559243e+06,1.569904e+06,1.569956e+06,1.574610e+06,1.570219e+06,1.586206e+06,1.590063e+06
19,Food PCE,DFSARC,2012-Jul,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.537353e+06,1.547922e+06,1.554310e+06,1.564078e+06,1.562783e+06,1.563877e+06,1.560248e+06,1.564161e+06,1.566873e+06,1.573443e+06
25,Food PCE,DFSARC,2022-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.575475e+06,1.575224e+06,1.598615e+06,1.602467e+06,1.616971e+06,1.610702e+06,1.607609e+06,1.659879e+06,1.646147e+06,1.649659e+06
2,Gasoline/Energy PCE,DGOERC,Historical Baseline,120816.0,117271.0,122232.0,122862.0,124918.0,124520.0,126837.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Gasoline/Energy PCE,DGOERC,2008-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.309007e+05,5.320788e+05,5.157662e+05,5.045685e+05,4.773657e+05,3.914478e+05,3.492894e+05,3.554018e+05,3.701022e+05,3.588865e+05
14,Gasoline/Energy PCE,DGOERC,2011-Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.287644e+05,5.186502e+05,5.136796e+05,5.103278e+05,5.147707e+05,5.072792e+05,5.082239e+05,5.020444e+05,5.029159e+05,5.171772e+05
20,Gasoline/Energy PCE,DGOERC,2012-Jul,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.496215e+05,5.255745e+05,5.061647e+05,5.200142e+05,5.327294e+05,5.174010e+05,5.037765e+05,5.024826e+05,5.043602e+05,5.069465e+05
26,Gasoline/Energy PCE,DGOERC,2022-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.261092e+05,4.864580e+05,4.603116e+05,4.491630e+05,4.594564e+05,4.550160e+05,4.305534e+05,4.382456e+05,4.395240e+05,4.316859e+05


In [15]:
# %%
# Cell 10: Direct Univariate Scenario Forecasting (Bypassing Intermediaries)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print(f"Executing Direct Univariate Scenario Forecasts for: {TARGET_SECTOR}...")

# 1. Configuration
HISTORICAL_WEIGHT = 0.80  
TIMESFM_WEIGHT = 1.0 - HISTORICAL_WEIGHT
print(f"Weighting Strategy: {HISTORICAL_WEIGHT*100}% Historical Scenario / {TIMESFM_WEIGHT*100}% TimesFM Baseline")

target_series = df_clean[TARGET_SECTOR].dropna()
target_last_date = target_series.index[-1]
baseline_value = target_series.iloc[-1]

# 2. Extract Direct Historical Trajectories
brent_above_100 = df_clean['Brent_Crude'] > 100
start_of_spikes = brent_above_100 & ~brent_above_100.shift(1, fill_value=False)
anchor_dates = df_clean.index[start_of_spikes]

target_shock_shapes = {}

for anchor in anchor_dates:
    scenario_name = anchor.strftime('%Y-%b')
    end_date = anchor + pd.DateOffset(months=12)
    
    if end_date <= target_series.index[-1]:
        slice_13m = target_series.loc[anchor:end_date]
        if len(slice_13m) >= 13: 
            trajectory = (slice_13m.values / slice_13m.values[0])[1:]
            target_shock_shapes[scenario_name] = trajectory
            print(f" - Extracted '{scenario_name}' historical trajectory for {TARGET_SECTOR}")

# 3. Generate TimesFM Baseline 
print("\nGenerating purely univariate TimesFM baseline...")
context_data = target_series.values[-CONTEXT_LENGTH:]

point_forecast, _ = tfm.forecast(
    horizon=HORIZON_LENGTH, 
    inputs=[context_data.astype(np.float32)]
)
timesfm_baseline = point_forecast[0][-HORIZON_LENGTH:]

# 4. Blend Baseline with Historical Scenarios
future_dates = pd.date_range(start=target_last_date + pd.DateOffset(months=1), periods=HORIZON_LENGTH, freq='MS')
direct_scenario_forecasts = {}

for scenario, shape in target_shock_shapes.items():
    pure_historical = baseline_value * shape
    blended_forecast = (pure_historical * HISTORICAL_WEIGHT) + (timesfm_baseline * TIMESFM_WEIGHT)
    
    df_scen = pd.DataFrame(index=future_dates)
    df_scen[f'{TARGET_SECTOR}_Direct_Forecast'] = blended_forecast
    direct_scenario_forecasts[scenario] = df_scen

print(f"✅ Generated {len(direct_scenario_forecasts)} direct scenario forecasts.\n")


# 5. Plotting (USING DYNAMIC CSV NAMES)
# FIX: Map from df_long instead of df_wide to avoid variable collision
csv_names = dict(zip(df_long['category_key'], df_long['category']))
target_display_name = csv_names.get(TARGET_SECTOR, TARGET_SECTOR)

historical_slice = target_series.loc['2005-01-01':]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=historical_slice.index, y=historical_slice.values,
    mode='lines', name=f'Historical {target_display_name}', 
    line=dict(color='black', width=3)
))

colors = px.colors.qualitative.Plotly 
for idx, scenario in enumerate(target_shock_shapes.keys()):
    color = colors[idx % len(colors)]
    
    plot_target = direct_scenario_forecasts[scenario].copy()
    plot_target.loc[target_last_date] = baseline_value
    plot_target = plot_target.sort_index()
    
    fig.add_trace(go.Scatter(
        x=plot_target.index, y=plot_target[f'{TARGET_SECTOR}_Direct_Forecast'],
        mode='lines', name=f'{scenario} Scenario', 
        line=dict(color=color, width=2.5, dash='dash')
    ))

fig.add_vline(x=target_last_date, line_width=2, line_dash="dot", line_color="black")

fig.update_layout(
    title=f"<b>Direct Univariate Forecast for {target_display_name}</b><br><sup>Blended strictly with historical >$100 Oil period behaviors</sup>",
    height=600, template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

Executing Direct Univariate Scenario Forecasts for: DVAPRC...
Weighting Strategy: 80.0% Historical Scenario / 19.999999999999996% TimesFM Baseline
 - Extracted '2008-Mar' historical trajectory for DVAPRC
 - Extracted '2011-Feb' historical trajectory for DVAPRC
 - Extracted '2012-Jul' historical trajectory for DVAPRC
 - Extracted '2022-Mar' historical trajectory for DVAPRC

Generating purely univariate TimesFM baseline...


✅ Generated 4 direct scenario forecasts.



In [16]:
# %%
# Cell 11: Export Direct Univariate Forecast Data (Horizontal Format)
import pandas as pd

print(f"Transforming and exporting univariate scenario forecasts for {TARGET_SECTOR}...")

# 1. FIX: Grab exact mapping from df_long to avoid variable collisions
csv_names = dict(zip(df_long['category_key'], df_long['category']))

# 2. Process Historical Target Data
df_hist_target = df_clean[[TARGET_SECTOR]].copy().T
df_hist_target['scenario'] = 'Historical Baseline'

# 3. Process Each Direct Forecast Scenario
scenario_target_dfs = []

for scenario_name in direct_scenario_forecasts.keys():
    df_tgt = direct_scenario_forecasts[scenario_name].rename(
        columns={f'{TARGET_SECTOR}_Direct_Forecast': TARGET_SECTOR}
    )
    
    df_combined = df_tgt.T
    df_combined['scenario'] = scenario_name
    scenario_target_dfs.append(df_combined)

# 4. Concatenate History and Scenarios Together
df_final_target = pd.concat([df_hist_target] + scenario_target_dfs)

# 5. Restructure the Metadata Columns
df_final_target.index.name = 'category_key'
df_final_target = df_final_target.reset_index()

# Insert the exact 'category' column pulled directly from the original CSV
df_final_target.insert(0, 'category', df_final_target['category_key'].map(csv_names).fillna(df_final_target['category_key']))

# 6. Clean up Date Headers 
date_cols = [col for col in df_final_target.columns if col not in ['category', 'category_key', 'scenario']]

formatted_date_cols = [d.strftime('%Y-%m') if pd.notna(d) else str(d) for d in date_cols]
rename_dict = dict(zip(date_cols, formatted_date_cols))
df_final_target = df_final_target.rename(columns=rename_dict)

final_col_order = ['category', 'category_key', 'scenario'] + formatted_date_cols
df_final_target = df_final_target[final_col_order]

df_final_target['sort_order'] = df_final_target['scenario'].apply(lambda x: 0 if x == 'Historical Baseline' else 1)
df_final_target = df_final_target.sort_values(by=['category_key', 'sort_order', 'scenario']).drop(columns=['sort_order'])

# 7. Export to CSV
export_filename = f"{TARGET_SECTOR}_Univariate_Horizontal_Forecasts.csv"
df_final_target.to_csv(export_filename, index=False)

print(f"✅ Data successfully exported to '{export_filename}'")
print(f"Rows: {len(df_final_target)} | Date Columns: {len(formatted_date_cols)}")

display(df_final_target.head())

Transforming and exporting univariate scenario forecasts for DVAPRC...
✅ Data successfully exported to 'DVAPRC_Univariate_Horizontal_Forecasts.csv'
Rows: 5 | Date Columns: 423


,category,category_key,scenario,1992-01,1992-02,1992-03,1992-04,1992-05,1992-06,1992-07,...,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12,2027-01,2027-02,2027-03
0,"Video, audio, photographic, and informat...",DVAPRC,Historical Baseline,59626.0,58998.0,58289.0,58321.0,58607.0,58857.0,59536.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Video, audio, photographic, and informat...",DVAPRC,2008-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,462775.254920,463538.759679,455839.304138,450007.511287,443723.766684,440875.102489,434901.832770,445899.447223,444506.047468,428535.110645
2,"Video, audio, photographic, and informat...",DVAPRC,2011-Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,459739.757319,462052.227700,462849.771563,463847.773577,459964.617457,473130.404187,468254.385208,462618.261110,463888.640911,469274.689198
3,"Video, audio, photographic, and informat...",DVAPRC,2012-Jul,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,452502.338134,460386.395509,462830.574719,464976.483131,466982.230254,461452.042672,459858.375233,460154.658558,461139.320944,461454.546396
4,"Video, audio, photographic, and informat...",DVAPRC,2022-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,453167.534914,457906.258216,460244.549329,462076.703887,461963.901367,447667.307015,452704.236840,474536.900009,474806.392462,475072.068973
